---
jupyter: python3
author: Corey T. White
execute:
  eval: true
  freeze: auto
date-modified: today
format:
  html:
    toc: true
    code-tools: true
    code-copy: true
    code-fold: false
---

# SSURGO Soil Data in GRASS

In [ ]:
# | label: imports
# | echo: false
import os
import subprocess
import sys

# import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
from PIL import Image
import pandas as pd
import sqlite3
from IPython.display import IFrame
import numpy as np
import seaborn as sns
import pandas as pd

# Ask GRASS GIS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

# Import the GRASS GIS packages we need.
import grass.script as gs

# Import GRASS Jupyter
import grass.jupyter as gj
from grass.tools import Tools

In [ ]:
# | label: init_session
gisdb = os.path.join(os.getenv("HOME"), "grassdata")
site = "coweeta"
mapset = "ssurgo_demo"
session = gj.init(gisdb, site, "PERMANENT")
tools = Tools(session=session)
try:
    tools.g_mapset(mapset=mapset, flags="c")
except Exception as e:
    print(f"{e}")

Install the `r.in.ssurgo` extension if it is not already installed. This extension is available in the GRASS Addons repository and can be installed using `g.extension`. The source code for this extension is also available in the `tools/r.soildb/r.in.ssurgo` directory of this repository.

In [ ]:
# | label: install_extension
# | eval: false
tools.g_extension(extension="r.in.ssurgo")

Set the computational region to the elevation raster and create a relief raster for hillshading.

In [ ]:
# | label: set_region
region_text = tools.g_region(raster="elevation", flags="p").text
tools.r_relief(input="elevation", output="relief")
print(region_text)

Import SSURGO data using `r.in.ssurgo`. Note that this will import the soil areas map, hydrologic group map, and ksat maps for low, regular, and high values. The `mukey` column is also imported as a raster with a categorical color scheme applied. The hydrologic group map also has a categorical color scheme applied.

In [ ]:
# | label: import_ssurgo
# | eval: false
tools.r_in_ssurgo(
    # ssurgo_path="../data/gSSURGO_CONUS.zip/gSSURGO_CONUS.gdb",
    soils="soil_areas",
    hydgrp="hydgrp",
    ksat_l="ksat_l",
    ksat_r="ksat_r",
    ksat_h="ksat_h",
    mukey="mukey",
)

In [ ]:
# | label: vector_maps
print(tools.g_list(type="vector", mapset=mapset).text)

In [ ]:
# | label: soil attributes
json_data = tools.v_db_select(map="soil_areas", format="json").json
df = pd.DataFrame(json_data["records"])
df.head()

In [ ]:
# | label: list_rasters
print(tools.g_list(type="raster", mapset=mapset).text)

## MUKEY Map

In [ ]:
# | label: mukey
m = gj.Map(use_region=True)
m.d_shade(shade="relief", color="mukey")
# m.d_legend(raster="mukey", title="MUKEY", flags="b")
m.show()

## Ksat Maps

In [ ]:
# | label: ksat
# | layout-ncol: 3
# | fig-cap: "Ksat maps for low, regular, and high values"
# | fig-subcap:
# |     - Low
# |     - Regular
# |     - High


m = gj.Map(use_region=True)
m.d_shade(shade="relief", color="ksat_l")
m.d_legend(raster="ksat_l", title="Ksat (mm/hr)", flags="b")
m.show()

m = gj.Map(use_region=True)
m.d_shade(shade="relief", color="ksat_r")
m.d_legend(raster="ksat_r", title="Ksat (mm/hr)", flags="b")
m.show()

m = gj.Map(use_region=True)
m.d_shade(shade="relief", color="ksat_h")
m.d_legend(raster="ksat_h", title="Ksat (mm/hr)", flags="b")
m.show()

## Hydrologic Group

In [ ]:
# | label: hydrologic_group_map
m = gj.Map(use_region=True)
m.d_rast(map="hsg")
m.d_shade(shade="relief", color="hsg")
m.d_legend(raster="hsg", title="Group", flags="b")
m.show()